# Study 890 — Sector Risk-Parity ⚖️

**Cap-weight buries the S&P in a few mega-cap sectors. If you equal-*risk*-weight the eleven
GICS sectors instead, do you get a better risk-adjusted ride?**

The pitch is "All-Weather, but *within* equities": weight each sector by inverse volatility
(or full equal-risk-contribution), rebalance quarterly, and see whether the diversification
lifts the **excess-of-cash Sharpe** and cuts the drawdown versus **cap-weight SPY**, net of
costs. Diversification, not forecasting.

We test two real panels — an **eleven-sector** headline (2018-10-01 → 2026-06-30, short
because XLC only launched 2018-06) and a longer **nine-sector** panel back to 2007-10-01
(BIL's inception) — both on yfinance daily total-return prices, everything excess of BIL cash.

*Numbers below are the frozen headline (`docs/results.md`); the only live cell runs the fast
synthetic control. Short history on the eleven-sector panel is named on the Signal axis.*


## 1. The idea in one picture

Today a third of SPY *is* Information Technology — cap-weight lets the biggest, highest-vol sectors dominate the portfolio's risk. Inverse-vol weighting hands each sector the **same risk budget**: it trims the crowded high-vol sectors and lifts the sleepy low-vol ones (staples, utilities, health care). The hope is a smoother ride at a better Sharpe. The catch: when the crowded sector (tech) is exactly what powers the bull market, *under*-weighting it costs you return.

In [1]:
R = {'e_start': '2018-10-01', 'e_end': '2026-06-30', 'e_n': 1946, 'e_rebals': 31, 'e_fp': '5e98273e423e', 'e_rows': 2018, 'e_rp_sharpe': 0.572, 'e_rp_ann': 12.73, 'e_rp_vol': 17.74, 'e_rp_dd': -34.7, 'e_spy_sharpe': 0.667, 'e_spy_ann': 15.67, 'e_spy_vol': 19.62, 'e_spy_dd': -33.7, 'e_diff': -0.095, 'e_ci': (-0.302, 0.132), 'e_pneg': 0.81, 'e_nwt': -1.5, 'e_turn': 57, 'e_cost': 1.7, 'e_erc_sharpe': 0.583, 'e_erc_diff': -0.084, 'n_start': '2007-10-01', 'n_end': '2026-06-30', 'n_n': 4716, 'n_rebals': 75, 'n_fp': '2afe10148f92', 'n_rows': 4802, 'n_rp_sharpe': 0.555, 'n_rp_ann': 11.16, 'n_rp_vol': 17.76, 'n_rp_dd': -49.6, 'n_spy_sharpe': 0.553, 'n_spy_ann': 12.29, 'n_spy_vol': 19.88, 'n_spy_dd': -55.2, 'n_diff': 0.002, 'n_ci': (-0.099, 0.11), 'n_pneg': 0.47, 'n_nwt': -1.12, 'n_turn': 48, 'n_cost': 1.4, 'n_erc_sharpe': 0.517, 'n_erc_diff': -0.036, 'era_early': '2007-2015', 'era_early_rp': 0.414, 'era_early_spy': 0.349, 'era_early_diff': 0.064, 'era_early_n': 2079, 'era_late': '2016-2026', 'era_late_rp': 0.695, 'era_late_spy': 0.759, 'era_late_diff': -0.064, 'era_late_n': 2637, 'lev_L': 1.12, 'lev_sharpe': 0.552, 'lev_spy_sharpe': 0.553, 'lev_ann': 10.97, 'lev_spy_ann': 10.99, 'lev_fin': 7.2, 'lev_dd': -54.2, 'cy_2008': 4.6, 'cy_2022': 13.3, 'cy_2023': -14.6, 'cy_2024': -10.6, 'cy_wins': 7, 'cy_years': 20, 'null_mean': 0.0031, 'null_sd': 0.0214, 'null_fire': 0, 'planted': 0.136, 'planted_rp': 1.32, 'planted_spy': 1.18}
print('ELEVEN-SECTOR (2018-2026, tech-led):')
print(f"  risk-parity  Sharpe {R['e_rp_sharpe']:.2f}  ann {R['e_rp_ann']:+.1f}%  maxDD {R['e_rp_dd']:.0f}%")
print(f"  cap-weight SPY Sharpe {R['e_spy_sharpe']:.2f}  ann {R['e_spy_ann']:+.1f}%  maxDD {R['e_spy_dd']:.0f}%")
print(f"  -> Sharpe difference {R['e_diff']:+.3f}  (95% CI {R['e_ci']}) - RP LOST over the tech bull")
print()
print('NINE-SECTOR (2007-2026, includes 2008):')
print(f"  risk-parity  Sharpe {R['n_rp_sharpe']:.2f}  ann {R['n_rp_ann']:+.1f}%  vol {R['n_rp_vol']:.1f}%  maxDD {R['n_rp_dd']:.0f}%")
print(f"  cap-weight SPY Sharpe {R['n_spy_sharpe']:.2f}  ann {R['n_spy_ann']:+.1f}%  vol {R['n_spy_vol']:.1f}%  maxDD {R['n_spy_dd']:.0f}%")
print(f"  -> Sharpe difference {R['n_diff']:+.3f}  (95% CI {R['n_ci']}) - a DEAD HEAT on Sharpe...")
print(f"     ...but RP cut vol ({R['n_rp_vol']:.1f}% vs {R['n_spy_vol']:.1f}%) and drawdown ({R['n_rp_dd']:.0f}% vs {R['n_spy_dd']:.0f}%)")

ELEVEN-SECTOR (2018-2026, tech-led):
  risk-parity  Sharpe 0.57  ann +12.7%  maxDD -35%
  cap-weight SPY Sharpe 0.67  ann +15.7%  maxDD -34%
  -> Sharpe difference -0.095  (95% CI (-0.302, 0.132)) - RP LOST over the tech bull

NINE-SECTOR (2007-2026, includes 2008):
  risk-parity  Sharpe 0.56  ann +11.2%  vol 17.8%  maxDD -50%
  cap-weight SPY Sharpe 0.55  ann +12.3%  vol 19.9%  maxDD -55%
  -> Sharpe difference +0.002  (95% CI (-0.099, 0.11)) - a DEAD HEAT on Sharpe...
     ...but RP cut vol (17.8% vs 19.9%) and drawdown (-50% vs -55%)


## 2. The tell — the 'advantage' flips sign every era

Split the long panel in half and the story is stark. In the crisis-heavy first half risk-parity **won** (2007-2015: Sharpe 0.41 vs 0.35, diff **+0.064**); in the tech-led second half it **lost** (2016-2026: 0.69 vs 0.76, diff **-0.064**). The two cancel to ~zero over the full sample. A real Sharpe edge should not swap signs with the regime — this is diversification, not alpha.

In [2]:
print(f"{R['era_early']}: RP {R['era_early_rp']:.3f}  SPY {R['era_early_spy']:.3f}  diff {R['era_early_diff']:+.3f}  (RP wins the crises)")
print(f"{R['era_late']}: RP {R['era_late_rp']:.3f}  SPY {R['era_late_spy']:.3f}  diff {R['era_late_diff']:+.3f}  (RP lags the tech bull)")
print()
print(f"Calendar-year: RP beat SPY by {R['cy_2008']:+.1f}pp in 2008 and {R['cy_2022']:+.1f}pp in 2022 (bear years),")
print(f"               but lagged by {R['cy_2023']:.1f}pp in 2023 and {R['cy_2024']:.1f}pp in 2024 (AI melt-up).")

2007-2015: RP 0.414  SPY 0.349  diff +0.064  (RP wins the crises)
2016-2026: RP 0.695  SPY 0.759  diff -0.064  (RP lags the tech bull)

Calendar-year: RP beat SPY by +4.6pp in 2008 and +13.3pp in 2022 (bear years),
               but lagged by -14.6pp in 2023 and -10.6pp in 2024 (AI melt-up).


## 3. Is the machinery even honest? A live synthetic control

Before trusting any of that, check the detector on a toy world where we *know* the answer. Every asset has the same Sharpe but different vols; the cap-weight benchmark piles on the high-vol names. When vols are dispersed, inverse-vol *should* out-Sharpe cap-weight; when all vols are equal it *must* tie. No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from sector_rp import data, strategy as st
null = st.synthetic_detect(data.synthetic_world(vol_spread=0.0, seed=890))
planted = st.synthetic_detect(data.synthetic_world(vol_spread=0.02, seed=890))
print('null world   (equal vols): Sharpe advantage %+.3f  (should be ~0)' % null['sharpe_advantage'])
print('planted world(dispersed) : Sharpe advantage %+.3f  (should be clearly +)' % planted['sharpe_advantage'])

null world   (equal vols): Sharpe advantage +0.002  (should be ~0)
planted world(dispersed) : Sharpe advantage +0.136  (should be clearly +)


## 4. The honest verdict

On the real tape the promised **Sharpe improvement is not there**: the nine-sector book ties SPY (diff **+0.002**, 95% CI (-0.099, 0.11) straddles zero) and the eleven-sector book *loses* over 2018–2026 (-0.095). What *is* real is the **risk reduction** — lower vol (17.8% vs 19.9%) and a milder drawdown (-50% vs -55%), with RP winning every genuine bear year. So the drawdown half of the claim holds and the Sharpe half does not: **Signal — Mixed**. And you cannot bank it: unlevered you simply earn *less* than SPY for the smoother ride, and levering the book back to SPY's vol just reproduces SPY (0.55 vs 0.55 Sharpe) — **Tradability — Mirage**.